<a href="https://colab.research.google.com/github/Kornieks/NYC-Job-Postings/blob/main/NYC_AI_skills_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
%cd /content/drive/MyDrive/Datasets

jobs_df = pd.read_csv('Jobs_NYC_Postings.csv')
jobs_df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Datasets


,Job ID,Agency,Posting Type,# Of Positions,Business Title,Civil Service Title,Title Classification,Title Code No,Level,Job Category,...,Additional Information,To Apply,Hours/Shift,Work Location 1,Recruitment Contact,Residency Requirement,Posting Date,Post Until,Posting Updated,Process Date
0,767684,DEPT OF ENVIRONMENT PROTECTION,Internal,1,"SECTION MANAGER, QUALITY ASSURANCE",ADMINISTRATIVE ENGINEER,Competitive-1,10015,M3,"Administration & Human Resources Engineering, ...",...,NaN,NaN,NaN,96-05 Horace Harding Expway,NaN,New York City Residency is not required for th...,02/02/2026,03-APR-2026,02/02/2026,03/17/2026
1,774000,DEPARTMENT OF BUILDINGS,Internal,1,Plan Examination Intern (Manhattan),SUMMER COLLEGE INTERN,Non-Competitive-5,10234,00,"Engineering, Architecture, & Planning",...,NaN,NaN,NaN,"280 Broadway, 3rd Floor, N.Y.",NaN,This internship does not require residency in ...,03/13/2026,27-MAR-2026,03/16/2026,03/17/2026
2,765029,NYC HOUSING AUTHORITY,Internal,1,Housing Assistant,HOUSING ASSISTANT (HA),Competitive-1,80201,00,Administration & Human Resources,...,1.\tFor NYCHA employees: this position is open...,NaN,NaN,LHD-Client Services Admin,NaN,NYCHA has no residency requirements.,02/27/2026,19-MAR-2026,02/27/2026,03/17/2026
3,770645,OFFICE OF THE MAYOR,External,1,Deputy Borough Director - Bronx,Special Assistant (MA)-MGRL,Pending Classification-2,0668A,M6,Constituent Services & Community Programs,...,NaN,NaN,NaN,253 Broadway New York Ny,NaN,New York City residency is generally required ...,02/17/2026,18-APR-2026,02/17/2026,03/17/2026
4,773101,HOUSING PRESERVATION & DVLPMNT,Internal,2,Agency Attorney L2 for the Housing Litigation ...,AGENCY ATTORNEY,Non-Competitive-5,30087,02,Legal Affairs,...,NaN,NaN,NaN,100 Gold Street,NaN,New York City Residency is not required for th...,03/09/2026,08-APR-2026,03/09/2026,03/17/2026


In [4]:
 api_key= 'Deleted for privicy purposes'

In [ ]:
template = '''You are an expert information extraction system.

Task:
Extract the skills mentioned in the required-skills paragraph and return them in valid JSON only.

Instructions:
1. Read the paragraph carefully.
2. Identify explicit skills only. Do not invent missing skills.
3. Normalize similar wording where appropriate.
   - Example: "Python programming" -> "Python"
   - Example: "experience with SQL databases" -> "SQL"
4. Separate skills into these categories when possible:
   - technical_skills
   - tools_platforms
   - soft_skills
   - domain_knowledge
   - certifications
5. If a skill appears preferred but not required, put it in `"preferred_skills"`.
6. If a category has no items, return an empty array.
7. Remove duplicates.
8. Output valid JSON only, with no explanation.

JSON schema:
{
  "technical_skills": [],
  "tools_platforms": [],
  "soft_skills": [],
  "domain_knowledge": [],
  "certifications": [],
  "preferred_skills": []
}

Input paragraph:
"""
{{required_skills_paragraph}}
"""'''

In [ ]:
text_1 = 'Preferred Skills / Qualifications  1.  Strong verbal and written communication skills with the ability to explain complex information clearly. 2.  Knowledge of program policies, procedures, and applicable state, federal, and HUD regulations 3.  Proficiency in program management systems and case processing software. 4.  Advanced proficiency in Microsoft Office Suite, including Excel (formulas, data analysis) and Word (document formatting and reporting). 5.  Strong analytical and mathematical skills, including accurate income and household calculations. 6.  High attention to detail and strong organizational skills. 7.  Ability to interpret and apply regulations consistently and accurately. 8.  Effective time management skills with the ability to prioritize and manage multiple cases. 9.  Strong problem-solving skills and ability to recommend process improvements. 10. Ability to maintain confidentiality and handle sensitive information with discretion.'

In [ ]:
prompt = template.replace('{{required_skills_paragraph}}', text_1)

In [ ]:

# Using the api_key variable from your previous cell

from openai import OpenAI
client = OpenAI(api_key=api_key)

response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
  ]
)

print(response.choices[0].message.content)

In [ ]:
def get_skills(text):
  prompt = template.replace('{{required_skills_paragraph}}', text)
  response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}])

  return str(response.choices[0].message.content)


In [ ]:
from tqdm import tqdm

skills = {}

for item in tqdm(jobs_df['Preferred Skills']):
  if pd.isna(item):
    continue
  if item in skills:
    continue
  skills_json = get_skills(item)
  skills[item] = skills_json

In [ ]:
import pandas as pd

# Convert the dictionary to a DataFrame
skills_df = pd.DataFrame(skills.items(), columns=['Skill Description', 'Extracted Skills JSON'])

# Display the first few rows of the DataFrame
display(skills_df.head())

In [ ]:
# Save the DataFrame to a CSV file
skills_df.to_csv('extracted_skills.csv', index=False)

print("Skills dictionary saved to 'extracted_skills.csv'")